# Data Integration

## Summary

**NSCG 2023 + O*NET Work Values is the selected integration.** The completed table contains **6,849 respondents across 25 occupations**, with one row per person, nine personal importance items and six occupational ratings. A reviewed study-specific concordance and conservative supervision screen define this subset. [Integration notebook](nscg_onet_data_integration.ipynb).

The surveys remain separate. GSS provides occupation candidates but limited aligned personal measures. Gallup supports preference-versus-job-condition comparisons within one survey. NLSY97 supports longitudinal analysis, with occupation and preference alignment still unresolved.

The integrated table supplies occupational context. The personal and occupational scales are not calibrated to each other, so no overall fit score is defined. Exploratory associations have not established reliable incremental prediction.

## 1. Data sources and scope

Counts refer to the source samples or extracts listed below; analysis-specific screens are reported in the following sections.

| Source | Sample and outcome | Occupation classification | Use and limits |
|---|---|---|---|
| GSS 2024 | 3,986 respondents; 2,782 valid satisfaction responses, four categories | `OCC10`: Census 2010 | **Use:** Add occupational or regional context.<br>**Limit:** Ambiguous occupation links and limited corresponding personal skill/value measures. |
| NSCG 2023 | 79,800 employed respondents with valid `JOBSATIS` and nine `FAC*` importance items | `N3OCPR`: NCSES taxonomy | **Use:** Compare personal priorities with occupational Work Values.<br>**Limit:** Selected occupation subset; incomplete construct alignment. Public-file completeness can include survey imputation. |
| Gallup AJQS 2025 | 18,429 respondents; `Q4` satisfaction, 0–10 | `Q6`: broad survey-specific groups | **Use:** Compare reported preferences and working conditions for the same person.<br>**Limit:** Few preference dimensions; broad groups limit detailed O*NET matching. |
| NLSY97, 2023 extract | 7,680 job records from 5,713 people with satisfaction responses, five categories | Census 2002 | **Use:** Study satisfaction across employment changes.<br>**Limit:** One person can have several job records; O*NET mapping and comparable personal preferences remain unresolved. |
| O*NET 31.0 / 30.2 | Occupation-level ratings; coverage varies by table | O*NET-SOC 2019 | **Use:** Skills and Work Context from 31.0; six Work Values from 30.2.<br>**Limit:** Ratings describe occupations, not individuals or specific workplaces. Most Work Values ratings date to 2008. |

Source documentation: [NSCG](https://ncses.nsf.gov/surveys/national-survey-college-graduates/2023#technical-notes), [Gallup AJQS](https://www.gallup.com/analytics/691241/american-job-quality-study.aspx), [NLS public releases](https://www.nlsinfo.org/content/access-data-investigator/accessing-data-cohorts), [O*NET Work Values 30.2](https://www.onetcenter.org/dictionary/30.2/excel/work_values.html).

### Integration design

Survey respondents are not joined across datasets: there is no shared person identifier, and matching on occupation would pair unrelated people. Pooling survey rows would require separate harmonization of outcomes, populations and weights.

Each survey can instead link to an occupation lookup. The survey-to-O*NET coding relationship must be established first; identical-looking codes from different classifications are not interchangeable. Several O*NET candidates indicate an ambiguous category link, not several occupations held by that person.

The NSCG implementation uses a many-to-one left join and preserves one row per eligible respondent. The analysis export contains the reviewed subset; the audit records all source respondents and inclusion decisions.

<a id="nscg"></a>
## 2. NSCG + O*NET integration

### 2.1 Occupation mapping and output

Use reviewed NSCG occupation `N3OCPR`, a study-specific NSCG-to-SOC concordance, and the [official SOC 2018-to-O*NET-SOC 2019 bridge](https://www.onetcenter.org/taxonomy/2019/soc.html). The 25 occupation links agree with published labels, questionnaire entries and occupation descriptions; individual duties cannot be independently recoded from the public file.

| Stage | Respondents |
|---|---:|
| Employed with valid satisfaction | 79,800 |
| In the 25 reviewed occupation categories | 7,017 |
| Held for supervision uncertainty | 168 |
| Integrated subset | **6,849** |

The supervision screen retains only `SUPWK = N` for sales engineers and bookkeeping clerks. NSCG and SOC apply different supervisor boundaries, and the public file does not resolve supervisory time shares. The screen may exclude valid matches.

The completed export has **6,849 unique respondents and 42 columns**: 24 NSCG fields, six Work Values and 12 mapping/provenance fields. All six ratings are present; source values are preserved. The left join retains 79,800 eligible rows before subset selection, and the separate audit covers all 94,606 source respondents.

This non-random subset represents **8.6% of eligible respondents**. The other occupations are outside the reviewed concordance, not necessarily unmatchable. The concordance is not an official NSCG-to-SOC crosswalk or a measured accuracy rate. Of the subset's 150 source ratings, 132 are dated June 2008 and 18 November 2020.

### 2.2 Preference alignment and exploratory associations

NSCG `FAC*` items measure personal importance on four ordered categories. O*NET `EX` measures how much an occupation can satisfy a Work Value on a 1–7 scale. Rescaling alone does not establish a common unit.

| NSCG item | O*NET Work Value | Alignment | Primary interaction OR (95% CI) |
|---|---|---|---|
| Independence (`FACIND`) | Independence | Closest exploratory pair; O*NET also includes creativity and responsibility | **1.17 (0.59–2.32)** |
| Advancement (`FACADV`) | Recognition | Advancement is one part of a broader value | **1.01 (0.77–1.32)** |
| Contribution to society (`FACSOC`) | Relationships | Social service is one part of a broader value | **1.03 (0.90–1.19)** |

Weighted logistic models use the same 6,849 people, with satisfied vs. dissatisfied as the primary outcome and uncertainty clustered by 25 occupations. ORs compare very vs. somewhat important, per one O*NET point. None of the three primary joint interaction tests is significant at 5% after Holm adjustment.

For the secondary **very satisfied** outcome, FACSOC × Relationships has OR **1.19 (1.07–1.33)**. Its direction persists across occupation omissions, but the interval includes 1 when medical and health services managers are omitted. Independence and advancement do not show consistent positive interactions across checks.

The other six NSCG items lack sufficiently specific counterparts in this table. Separate salary and security ratings are unavailable, and Support has no corresponding NSCG item. These exploratory models use approximate intervals, not full survey-design inference. Construct differences, occupation composition and old ratings limit a fit interpretation.

## 3. GSS + O*NET feasibility

### 3.1 Occupation candidates

The analysis uses **2,282 working respondents with valid `SATJOB`**, without imputing satisfaction or occupation. Counts are unweighted.

The `OCC10 → Census 2018 → SOC 2018 → O*NET` route yields at least one candidate for **2,201 people**: **1,013** have one and **1,188** have several. The remaining 81 comprise 76 missing occupation codes and five without a detailed SOC correspondence. No unresolved parser gaps remain in this audit.

Among multi-candidate records, **290** pass a similarity screen for at least one skills/value domain; none pass for the full profile. The screen requires complete usable profiles in one SOC major group, mean candidate spread ≤10 and maximum spread ≤20 on a 0–100 scale. Level ratings also require agreement on relevance. These are analyst-selected cutoffs, not evidence that candidates are equivalent.

Occupation candidates provide potential context links. GSS lacks a complete corresponding personal skill/value profile, so these ratings alone do not measure person-job fit.

### 3.2 Education requirements

Candidate education-profile screening identifies **681 people**. Requiring fully allocated occupational requirements after alignment to five broad GSS degree bins retains **141**: 20.7% of those screened, or 6.2% of the working sample. All 141 have one occupation candidate. The other 540 have partly unaligned requirements, rather than missing personal degree data.

The alignment pools some college with high school and combines graduate degree types; certificates and postdoctoral training remain unallocated. Allowing up to 5% unallocated share raises coverage to **297**, including 156 additional partial comparisons.

These counts apply to the education-alignment route. They do not describe overall GSS-to-O*NET matchability or establish education fit.

### 3.3 Alternative keys

| Key | Coverage | Interpretation |
|---|---|---|
| `ISCO08` | Both occupation codes cover 2,206 people; 323 `OCC10` categories correspond to 248 `ISCO08` categories | No additional detail or coverage in this sample; an ISCO-to-O*NET bridge was not constructed. |
| `INDUS10` | 2,184 people have codes listed in the Census industry classification; 98 lack a code | Verifies industry definitions, not links to external indicators or personal fit. |
| Region + 2024 | All 2,282 people have a key for BLS regional unemployment and quits lookups, across four regions | Regional context only. In one year, rates add no information beyond region indicators. |

Industry statistics would require aligned categories and employer coverage. Regional rates remain in separate lookups; monthly rates were not linked to respondents. Neither alternative supplies the missing individual preference profile.

## 4. Gallup: within-survey comparisons

Gallup contains personal preferences, reported job conditions and satisfaction for the same respondent. These comparisons require feature construction within the survey, rather than an external occupation join.

### 4.1 Promotion interest and opportunity

**13,082 employees** have valid promotion interest (`Q33H`), opportunity (`Q34`) and satisfaction (`Q4`). A weighted model of satisfaction (0–10) includes interest, opportunity and their interaction.

The opportunity gap is **+1.31 points** among interested employees and **+1.06** among those not interested. The difference is **+0.24 (95% CI −0.07 to +0.55)**; unweighted and adjusted comparisons remain inconclusive.

Keeping all five interest responses reveals interaction differences (joint p = .00084), also present without weights and with basic controls. The opportunity gaps do not increase consistently with interest. The pattern therefore supports group differences, without a simple increasing fit relationship. Both measures concern the current job and are contemporaneous self-reports.

### 4.2 Actual and preferred WFH days

**14,637 respondents** have valid actual and preferred WFH days (0–7), satisfaction, and 1–7 days worked, with actual WFH days no greater than days worked. Responses stating that WFH is impossible are excluded. Actual days refer to days worked **exclusively from home last week**; preferred days refer to the desired weekly arrangement.

Separate shortfall and excess terms allow different associations on either side of the preference. With survey weights, preferred days, worked days and basic controls, each shortfall day is associated with **−0.26 satisfaction points (95% CI −0.30 to −0.22)**. Estimates remain similar without weights, among employees and in five-day workweeks.

The excess-day estimate is **−0.14 (−0.26 to −0.02)**; employee and five-day checks include zero. Two-day and 3+-day shortfalls are clearer than a one-day gap. Actual days cover only one week, and preferences may reflect current satisfaction, so these associations do not establish causality.

### 4.3 WFH prediction

A fixed 80/20 split uses **11,709 training and 2,928 test respondents**. The weighted baseline includes numeric actual days, categorical preferred days and basic controls. Adding an absolute-gap term allows asymmetric mismatch. A second comparison treats both day counts as categorical.

| Weighted test metric | Baseline | With mismatch |
|---|---:|---:|
| RMSE | 2.13476 | 2.12954 |
| MAE | 1.64596 | 1.63947 |
| R² | 0.06250 | 0.06707 |

RMSE falls by **0.24%**. The difference is **−0.00521 (paired-bootstrap 95% CI −0.01337 to +0.00230)**. The categorical-day baseline gives a similar small, uncertain gain; training-only CV improves in four of five folds.

Full-sample exploration preceded the split, so this is exploratory internal validation. Bootstrap intervals condition on fitted models and omit training and full survey-design uncertainty. The results do not establish reliable incremental prediction beyond actual and preferred days.

## 5. NLSY97: longitudinal scope

The public 1997–2023 release supports repeated observations and employer identifiers. Restricting to a current main job with valid satisfaction and occupation yields **4,806 people in 2023** and **3,791 observed in both 2021 and 2023**. These counts precede any O*NET linkage.

NLSY97 can support questions about satisfaction around employment changes. Census 2002-to-O*NET mapping remains unaudited, and the catalog search did not identify a work-value battery comparable to NSCG `FAC*`. Employer and occupation changes also require validation before they are treated as job transitions. Longitudinal availability alone does not resolve preference alignment.